## Install and import required libraries

In [1]:
# !pip install scikit-learn
# !pip install numpy
# !pip install pandas
# !pip install matplotlib
# !pip install seaborn
# !pip install xgboost
# !pip install pyarrow

In [2]:
import pandas as pd
import numpy as np
import glob
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb

In [3]:
#load single file to look at contents
df = pd.read_csv('data/ids_0.csv')

#get names of columns
print(df.columns)

#display data types
print(df.dtypes)

#disply dataframe head
df.head()

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,53,61205,4,2,136,428,34,34,34.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,53,222,2,2,90,172,45,45,45.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,53,23759,2,2,70,126,35,35,35.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,401,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,57406,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
#combine all csv, json, and parquet files into single dataframe

def extract() -> pd.DataFrame:
    #main dataframe that everything will be concatenated to
    data = pd.DataFrame()

    #extract CSV files
    for csvfile in glob.glob('data/*.csv'):
        tmp_df = pd.read_csv(csvfile)
        data = pd.concat([data, tmp_df], ignore_index=True)
    
    #extract JSON files
    for jsonfile in glob.glob('data/*.json'):
        tmp_df = pd.read_json(jsonfile, lines=True)
        data = pd.concat([data, tmp_df], ignore_index=True)

    #extract Parquet files
    for parquetfile in glob.glob('data/*.parquet'):
        tmp_df = pd.read_parquet(parquetfile)
        data = pd.concat([data, tmp_df], ignore_index=True)
    
    #return combined dataframe
    return data

In [5]:
#call extract function to combine all files into single dataframe
df = extract()

print("Shape of combined dataframe:", df.shape)

df.head()

Shape of combined dataframe: (63129, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,55109,17,1,1,6,6,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,53,113594958,4,4,152,362,45,31,38.0,8.082904,...,32,240.0,0.0,240,240,114000000.0,0.0,114000000,114000000,BENIGN
2,53,30485,1,1,81,209,81,81,81.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,53,30445,1,1,53,81,53,53,53.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,53,70860,1,1,56,72,56,56,56.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [6]:
#verify the extracted data
#remove leading/trailing whitespace from column names
df.columns = df.columns.str.strip()  

#check label distribution
print(df['Label'].value_counts())

Label
DoS Hulk            31027
DoS GoldenEye       20586
BENIGN               6006
DoS Slowhttptest     5499
Heartbleed             11
Name: count, dtype: int64


In [8]:
#Tranform 

def transform(df: pd.DataFrame) -> pd.DataFrame:

    #remove Heartbleed rows since its not a DOS attack
    df = df[df['Label'] != 'Heartbleed']

    #remove any duplicate rows
    df = df.drop_duplicates()

    #remap labels (benign stays but all dos attacks get remapped to "attack")
    df['Label'] = df['Label'].apply(lambda x: 'BENIGN' if x == 'BENIGN' else 'attack')

    #replace infinite values with NaN and then drop rows with NaN values
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()

    return df


In [9]:
#call transform function to clean the data
df = transform(df)
print("Shape of transformed dataframe:", df.shape)

#check label distribution after transformation
print(df['Label'].value_counts())

Shape of transformed dataframe: (49348, 79)
Label
attack    44402
BENIGN     4946
Name: count, dtype: int64


In [ ]:
#load the data into a csv file
df.to_csv('data/cleaned_data.csv', index=False)